# Long-term capacity of a driven pile from its driving log

The `driven-pile` package reads a pile-driving record (how many hammer blows it took to advance the pile each foot) and estimates how much load the pile can carry. Every blow is treated as a cycle of shear on the sand near the pile toe. Enough cycles raise the pore water pressure until the sand liquefies, which is why a pile driven into saturated sand carries *less* at the end of driving than it will later, once the excess pore pressure has drained away. The package reports both numbers: the **end-of-drive** capacity and the **long-term** capacity. The difference is called *setup*.

This notebook installs the package on JupyterHub and walks through its quick start. The last section shows where a time-dependent model of pore pressure dissipation would plug in, which is the opening for a project on *how long* the setup takes.

Documentation: <https://lornearnold.github.io/driven-pile/> (the quick start, a walkthrough on a real log, and a page mapping each equation to the function that implements it). Units throughout are US customary: feet, kips (1000 lb) and ksf (kips per square foot).

## Install

Run the cell below once. It installs the package into your JupyterHub account (it is not there by default). If the `import` in the next cell fails right after installing, restart the kernel (Kernel menu, Restart Kernel) and run the import again.

In [ ]:
%pip install --quiet "driven-pile[plot]"

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import driven_pile as dp

print("driven-pile", dp.__version__)

## The quick start

### A short driving log

A driving log records, for each foot the pile advanced, the hammer's blow rate (blows per minute, which sets the stroke of a diesel hammer) and the number of blows it took (blows per foot). Here is a 20 ft log, typed in as arrays. The pile is a 24 in octagonal concrete pile and the hammer has a 20 kip ram.

`DriveLog.from_arrays` needs to know where each foot is. Giving it the depth of each logged foot and the toe elevation at the end of the drive is enough; here the log starts 20 ft above the final toe elevation of -50 ft.

In [ ]:
bpm = np.array([55, 55, 53, 48, 42, 42, 50, 50, 49, 46, 46, 50, 49, 46, 46, 46, 50, 49, 46, 46])
bpf = np.array([3, 5, 5, 7, 4, 9, 13, 20, 12, 16, 28, 27, 33, 31, 21, 18, 19, 22, 35, 40])

pile = dp.Pile.octagonal(width_in=24, length_ft=60)
hammer = dp.Hammer(ram_weight=20.0)  # kips

log = dp.DriveLog.from_arrays(
    pile,
    hammer,
    bpm=bpm,
    bpf=bpf,
    depth=np.arange(1, len(bpm) + 1),  # ft below the first logged foot
    tip_elevation=-50.0,               # ft, toe elevation at the end of the drive
)
print(f"{len(log.elevation)} logged feet, from {log.elevation.max():.0f} to {log.elevation.min():.0f} ft")

### The soil

The soil profile needs the mudline (ground surface) elevation, the water table, and the fines content (percent passing the No. 200 sieve), which controls how easily the sand liquefies. A single number gives a uniform profile; the walkthrough in the docs shows how to pass fines content that varies with elevation.

Notice that the mudline is at -10 ft but the log starts at -31 ft. The 20 ft in between were driven without being logged (a common situation). The package treats this unlogged top segment specially: it tries a range of friction angles for it and keeps the lowest one consistent with the log.

In [ ]:
soil = dp.SoilProfile(mudline=-10.0, water_elevation=0.0, fines=15.0)

### Run the analysis

`analyze` does everything: it converts each foot's blow count to a capacity with a dynamic pile formula, smooths that over the log, then works down the pile foot by foot finding the friction angle of the soil and tracking the pore pressure that the blows build up. The plot compares the capacity implied by the formula at each foot (gray dots), the smoothed target the model tries to fit (blue), and what the model reproduces (orange, dashed).

In [ ]:
result = dp.analyze(log, soil)
drive = result.drive

fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(drive.raw_capacity, drive.elevation, ".", color="gray", label="formula per foot")
ax.plot(drive.target, drive.elevation, color="C0", label="smoothed target")
ax.plot(drive.predicted, drive.elevation, "--", color="C1", label="modeled")
ax.set_xlabel("capacity at time of driving (kips)")
ax.set_ylabel("elevation (ft)")
ax.legend(loc="upper right", framealpha=1)
ax.grid(True, alpha=0.3)
plt.show()

### End of drive versus long term

`drive` holds one value per foot, from the mudline to the toe, of everything the model worked out: friction angle (`phi`), pore pressure ratio at the end of driving (`ru`, where 1 means fully liquefied), whether and when the foot liquefied (`liquefied`, `liq_when`), and the unit shaft resistance at the end of the drive (`fs_eod`) and after the pore pressure has dissipated (`fs_ltc`). The shaded area between the two shaft resistance curves, times the pile perimeter, is the setup.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(drive.fs_eod, drive.elevation, color="C0", label="end of drive")
ax.plot(drive.fs_ltc, drive.elevation, color="C1", label="long term")
ax.fill_betweenx(drive.elevation, drive.fs_eod, drive.fs_ltc, color="C1", alpha=0.25, label="setup")
ax.set_xlabel("unit shaft resistance (ksf)")
ax.set_ylabel("elevation (ft)")
ax.legend(loc="upper right", framealpha=1)
ax.grid(True, alpha=0.3)
plt.show()

Where did the setup come from? Plot the pore pressure ratio. The feet that liquefied carry almost nothing at the end of the drive, and they are the feet that gain the most once they drain.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(drive.ru, drive.elevation, color="C3")
ax.set_xlabel(r"pore pressure ratio $r_u$ at end of drive")
ax.set_ylabel("elevation (ft)")
ax.set_xlim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.show()

print(f"{100 * drive.fraction_liquefied():.0f}% of the logged feet liquefied during the drive")

### Adding it up

The shaft carries the unit resistance of each foot times the pile perimeter, accumulated from the mudline down. The toe resistance is added at the bottom. The table at the end is the summary a designer would want. The long-term total should equal `drive.ltc`.

In [ ]:
perimeter = drive.pile.perimeter  # ft
shaft_eod = np.nancumsum(drive.fs_eod) * perimeter  # kips, one foot per entry
shaft_ltc = np.nancumsum(drive.fs_ltc) * perimeter

fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(shaft_eod, drive.elevation, color="C0", label="shaft, end of drive")
ax.plot(shaft_ltc, drive.elevation, color="C1", label="shaft, long term")
ax.plot(drive.toe_eod, drive.elevation[-1], "o", color="k", label="toe")
ax.set_xlabel("resistance accumulated from the mudline (kips)")
ax.set_ylabel("elevation (ft)")
ax.legend(loc="upper right", framealpha=1)
ax.grid(True, alpha=0.3)
plt.show()

toe = drive.toe_eod
rows = [("end of drive", shaft_eod[-1], toe),
        ("long term", shaft_ltc[-1], toe),
        ("setup", shaft_ltc[-1] - shaft_eod[-1], 0.0)]
print(f"{'':14s}{'shaft':>8s}{'toe':>8s}{'total':>8s}   (kips)")
for name, shaft, toe_part in rows:
    print(f"{name:14s}{shaft:8.0f}{toe_part:8.0f}{shaft + toe_part:8.0f}")
print(f"\ndrive.ltc = {drive.ltc:.0f} kips")

## A real driving record

The two CSV files in this folder are the driving log and fines profile of pile B-20 from the walkthrough in the docs: a 170.5 ft octagonal pile driven 106 ft into the ground, with the first 32 ft unlogged. This log records the hammer stroke in feet instead of the blow rate, which `from_csv` accepts. Fines content varies with elevation, so the soil profile takes two arrays. Everything above works unchanged on `drive_b20`. Compare the end-of-drive and long-term numbers: this pile gains most of its capacity after driving stops, so *when* it gets there matters. The walkthrough page goes on to run the Monte Carlo analysis (`run_mc=True`), which puts a range on the long-term capacity instead of a single number.

In [ ]:
pile_b20 = dp.Pile.octagonal(width_in=24, length_ft=170.5)
log_b20 = dp.DriveLog.from_csv(
    "B-20_log.csv", pile_b20, hammer,
    elevation="elevation_ft", stroke="stroke_ft", bpf="bpf", pile_id="B-20",
)
fines = np.loadtxt("B-20_fines.csv", delimiter=",", skiprows=1)
soil_b20 = dp.SoilProfile(mudline=-50.0, elevation=fines[:, 0], fines=fines[:, 1])

drive_b20 = dp.analyze(log_b20, soil_b20).drive
print(f"{len(log_b20.elevation)} logged feet; {100 * drive_b20.fraction_liquefied():.0f}% liquefied during the drive")
print(f"end of drive {drive_b20.eod_capacity:.0f} kips, long term {drive_b20.ltc:.0f} kips "
      f"(shaft {drive_b20.ltc_shaft:.0f} + toe {drive_b20.toe_eod:.0f})")

fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
drive_b20.plot_fit(ax=axes[0])
drive_b20.plot_profile(ax=axes[1])
plt.tight_layout()
plt.show()